# LFRic Iris data manipulation and visualisation practical

Let's apply what we've learned in Section 01 to Section05 with the two exercises. The aim is to use the prompts to write the code yourself, but we have also provided a separate notebook containing the answers if you are stuck. All the information needed to write the code for this practical can be found in the notebooks in the first part of this practical.

Note: This is delivered in JupyterLabs, but sometime the PyVista and GeoVista plotting is laggy in labs. If you prefer you can run in IPython.

## Step 0: Conda setup (required for regridding)

Run these commands in a **fresh terminal** before running notebook cells:

```bash
cd <path to directory>/LFRic-Atmosphere-Training
conda create -n lfric-mesh python=3.12 -y
conda activate lfric-mesh
conda install -c conda-forge esmpy -y
# zsh-safe quoting for extras is important
python -m pip install -e '.[mesh_tutorials]'
python -m pip install jupyterlab
python -m pip show esmf-regrid
python -c "import esmf_regrid, ESMF; print('Imports OK')"
python -m jupyter lab --version
python -m ipykernel install --user --name lfric-mesh --display-name "Python (lfric-mesh)"
python -m jupyter lab
```

If `python -m pip show esmf-regrid` returns nothing, the package is not installed in that env.
If `import esmf_regrid` fails, rerun the quoted install command exactly as shown.

Then in Jupyter choose **Kernel -> Change Kernel -> Python (lfric-mesh)** and restart the kernel.

Reason: `esmf-regrid` needs the compiled ESMF backend (`esmpy`), which is installed from conda-forge.


In [1]:
# Notebook warning controls
import warnings

# Suppress repeated NumPy masked-array casting warnings from sample data loading.
warnings.filterwarnings(
    "ignore",
    message="invalid value encountered in cast",
    category=RuntimeWarning,
)
# Suppress Iris legacy date-precision future warning in tutorial output.
warnings.filterwarnings(
    "ignore",
    message="You are using legacy date precision for Iris units*",
    category=FutureWarning,
)

# Use microsecond date precision to align with upcoming Iris default behaviour.
try:
    import iris
    iris.FUTURE.date_microseconds = True
except Exception:
    pass

# Check that this kernel can run regridding
import importlib.util
import sys
print("Kernel Python:", sys.executable)
assert importlib.util.find_spec("ESMF") or importlib.util.find_spec("esmpy"), "ESMPy backend missing in this kernel"
assert importlib.util.find_spec("esmf_regrid"), "esmf_regrid missing in this kernel"
print("Backend check: OK")


Kernel Python: /Users/lb788/miniconda3/envs/lfric-mesh/bin/python
Backend check: OK


## Exercise 2 - Regrid UM data to LFRic and plot using PyVista

Now you can do a similar exercise compared to the previous [Exercise 01](./06_Exercise_01.ipynb), but regrid UM data onto a LFRic mesh and plot the data using PyVista

**Step 1** To begin, we need to import the neccesary packages that we will need for this exercise.

In [2]:
%matplotlib inline
import pyvista as pv
import geovista as gv
import geovista.theme
import iris.quickplot as qplt
import iris
from geovista import GeoPlotter
from esmf_regrid import ESMFAreaWeighted
pv.set_jupyter_backend("static")
iris.FUTURE.datum_support = True  # avoids some warnings

The [pv_conversions](./pv_vonversions.py) script contains two functions which convert LFRic cubes to pyvista objects. Load these two functions:

In [3]:
from support.pv_conversions import pv_from_lfric_cube
from support.pv_conversions import pv_from_um_cube

**Step 2** Lets chose a different diagnostic, 'surface_temperature' and load both the UM data, as well LFRic data to use as reference grid.

In [4]:
data_path = '../example_data/'
lfric_path = data_path + 'u-ct674_20210324T0000Z_lf_ugrid.nc'
um_path = data_path + 'u-ct674_20210324T0000Z_um_latlon.nc'

with iris.LOAD_POLICY.context(support_multiple_references=True):
    lfric_rho = iris.load_cube(lfric_path, 'surface_air_pressure')
    
um_temp = iris.load_cube(um_path, 'surface_temperature')
um_temp_t0 = um_temp[0]

**Step 3** Create a regridder from 'ESMFAreaWeighted' \
Then use the regridder to regrid the new UM cube created earlier. Print your result and notice the mesh characterisics.

In [5]:
# Continue here...

**Step 4** Plot the regridded UM data with PyVista. \
(hint: before you can do this you will need to convert you mesh to polydata using pv_from_lfric_cube as explained in [Section 03](./Sec_03_Plotting.ipynb))

**Step 5** Plot the native UM data with PyVista. \
(hint: before you can do this you will need to convert you mesh to polydata using pv_from_um_cube)

**Step 6** Now we can plot this data side by side \
(hints: start by using plotter = GeoPlotter(shape=(1,2)), then create your subplots, add your coastlines, add a base layer, and add your mesh) \
note: PyVista and GeoVista can be slow in jupyter labs, but try and move the plots, look at the poles - you might notice the polar sigularity problem of the lat-lon grid.

**Step 7** In notebook [Section 03](./Sec_03_plotting.ipynb), we see how to use the plotter.camera_position = viewpoint functionality. Try this out with the surface temperature data. 

**Finished!?** These two exercises were just the beginning. If you have time try adding some cells below and extract a zonal mean, or try to select a region of data. 